In [5]:
import pandas as pd
import sqlite3
from scipy import stats

# 1. Connect to your PROCESSED database
# Ensure the path matches your folder: '../Processed Data/processed_pharma_sfe.db'
conn = sqlite3.connect('../Processed Data/processed_pharma_sfe.db')

# 2. Load all 5 tables from the database
sales = pd.read_sql("SELECT * FROM sales", conn)
reps = pd.read_sql("SELECT * FROM reps", conn)
territories = pd.read_sql("SELECT * FROM territories", conn)
calls = pd.read_sql("SELECT * FROM calls_cleaned", conn)
prescribers = pd.read_sql("SELECT * FROM prescribers_cleaned", conn)

conn.close()
print("All tables loaded successfully into memory: sales, reps, territories, calls, prescribers")

All tables loaded successfully into memory: sales, reps, territories, calls, prescribers


In [2]:
# Check the distribution of A-tier focus
print("A-tier Focus Statistics:")
print(df_stats['a_tier_pct'].describe())

# Check how many reps fall into each group with the 40% threshold
focused_count = len(df_stats[df_stats['a_tier_pct'] > 0.4])
not_focused_count = len(df_stats[df_stats['a_tier_pct'] <= 0.4])

print(f"\nReps with >40% A-tier focus: {focused_count}")
print(f"Reps with <=40% A-tier focus: {not_focused_count}")

A-tier Focus Statistics:
count    500.000000
mean       0.229111
std        0.041319
min        0.114943
25%        0.200000
50%        0.226667
75%        0.253559
max        0.367089
Name: a_tier_pct, dtype: float64

Reps with >40% A-tier focus: 0
Reps with <=40% A-tier focus: 500


In [8]:
# 1. Use the median focus percentage as the new threshold
threshold = df_stats['a_tier_pct'].median()
print(f"Using Data-Driven Threshold (Median): {threshold:.2%}")

# 2. Split into two groups based on the median
focused = df_stats[df_stats['a_tier_pct'] > threshold]['achievement']
not_focused = df_stats[df_stats['a_tier_pct'] <= threshold]['achievement']

# 3. Verify sample sizes
print(f"Focused group size: {len(focused)}")
print(f"Not-focused group size: {len(not_focused)}")

# 4. Run the T-test again
t_stat, p_value = stats.ttest_ind(focused, not_focused)
print(f"\nRE-CALCULATED RESULTS:")
print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.4f}")

# 5. Interpret the result
if p_value < 0.05:
    print("CONCLUSION: Statistically Significant. Targeting A-tier doctors drives performance.")
else:
    print("CONCLUSION: Not Significant. Targeting strategy does not appear to be the main driver.")

Using Data-Driven Threshold (Median): 22.67%
Focused group size: 249
Not-focused group size: 251

RE-CALCULATED RESULTS:
T-statistic: 1.7622, P-value: 0.0787
CONCLUSION: Not Significant. Targeting strategy does not appear to be the main driver.


In [7]:
# 1. Prepare data: Join sales achievement with territory zones
zone_data = sales.merge(territories[['territory_id', 'zone']], on='territory_id')

# 2. Group achievement by zone
groups = [zone_data[zone_data['zone'] == z]['target_achievement_pct'] for z in zone_data['zone'].unique()]

# 3. Run One-Way ANOVA
f_stat, p_val_anova = stats.f_oneway(*groups)

print(f"ANOVA Results: F-Statistic = {f_stat:.4f}, P-value = {p_val_anova:.4f}")

if p_val_anova < 0.05:
    print("CONCLUSION: Significant regional variance. Strategy should be zone-specific.")
else:
    print("CONCLUSION: No significant regional variance. Strategy can be national.")

ANOVA Results: F-Statistic = 1.0177, P-value = 0.3840
CONCLUSION: No significant regional variance. Strategy can be national.


In [10]:
import sqlite3
import pandas as pd

def run_test(query_number, query_text):
    """Connects to the DB, runs a query, and displays the result."""
    conn = sqlite3.connect('../Processed Data/processed_pharma_sfe.db')
    try:
        print(f"--- Running Query {query_number} ---")
        result = pd.read_sql(query_text, conn)
        display(result.head(10)) # Shows top 10 rows
        print(f"Success! Query {query_number} returned {len(result)} rows.\n")
    except Exception as e:
        print(f"Error in Query {query_number}: {e}\n")
    finally:
        conn.close()

In [11]:
# Q1: Total revenue by territory, current year
q1 = """
SELECT territory_id, SUM(revenue) AS total_revenue
FROM sales 
GROUP BY territory_id 
ORDER BY total_revenue DESC;
"""

# Q2: Average target achievement % by region (Zone)
q2 = """
SELECT t.zone, AVG(s.target_achievement_pct) AS avg_achievement
FROM sales s 
JOIN territories t USING(territory_id)
GROUP BY t.zone;
"""

# Q5: Unvisited High-Value Prescribers (The "Low-Hanging Fruit" query)
# Note: This is a LEFT JOIN to find doctors with NULL call IDs
q5 = """
SELECT p.prescriber_id, p.specialty, p.prescriber_tier
FROM prescribers_cleaned p
LEFT JOIN calls_cleaned c ON p.prescriber_id = c.prescriber_id
      AND c.call_date >= DATE('now', '-90 days')
WHERE p.prescriber_tier = 'A' AND c.call_id IS NULL;
"""

# Run the tests
run_test(1, q1)
run_test(2, q2)
run_test(5, q5)

--- Running Query 1 ---


,territory_id,total_revenue
0,29,6.815974e+06
1,44,6.716229e+06
2,34,6.668986e+06
3,52,6.660913e+06
4,16,6.651719e+06
5,4,6.536514e+06
6,57,6.532261e+06
7,56,6.457964e+06
8,7,6.452851e+06
9,9,6.449764e+06


Success! Query 1 returned 80 rows.

--- Running Query 2 ---


,zone,avg_achievement
0,East,86.209414
1,North,84.930054
2,South,84.756871
3,West,83.976120


Success! Query 2 returned 4 rows.

--- Running Query 5 ---


,prescriber_id,specialty,prescriber_tier
0,DOC00003,Cardiologist,A
1,DOC00004,Cardiologist,A
2,DOC00015,GP,A
3,DOC00017,Cardiologist,A
4,DOC00028,Cardiologist,A
5,DOC00029,Diabetologist,A
6,DOC00036,GP,A
7,DOC00045,Cardiologist,A
8,DOC00058,GP,A
9,DOC00079,Diabetologist,A


Success! Query 5 returned 771 rows.



In [13]:
# Q6: Call efficiency (Rx share generated per call)
q6 = """
SELECT r.rep_id, 
       COUNT(c.call_id) AS total_calls,
       AVG(p.our_brand_rx_share) AS avg_rx_share,
       AVG(p.our_brand_rx_share) / NULLIF(COUNT(c.call_id), 0) AS rx_per_call
FROM reps r
JOIN calls_cleaned c ON r.rep_id = c.rep_id
JOIN prescribers_cleaned p ON c.prescriber_id = p.prescriber_id
GROUP BY r.rep_id
ORDER BY rx_per_call DESC;
"""

# Q8: Month-over-Month (MoM) revenue growth using Window Functions
q8 = """
SELECT territory_id, month, revenue,
       LAG(revenue) OVER (PARTITION BY territory_id ORDER BY month) AS prev_month_revenue,
       ROUND((revenue - LAG(revenue) OVER (PARTITION BY territory_id ORDER BY month)) * 100.0 / 
       NULLIF(LAG(revenue) OVER (PARTITION BY territory_id ORDER BY month), 0), 2) AS mom_growth_pct
FROM sales;
"""

# Run the tests
run_test(6, q6)
run_test(8, q8)

--- Running Query 6 ---


,rep_id,total_calls,avg_rx_share,rx_per_call
0,REP0268,58,29.314310,0.505419
1,REP0192,60,29.442000,0.490700
2,REP0176,66,31.369545,0.475296
3,REP0467,65,30.849846,0.474613
4,REP0398,66,31.307576,0.474357
5,REP0331,69,32.613188,0.472655
6,REP0313,64,30.120156,0.470627
7,REP0131,66,30.288485,0.458916
8,REP0034,66,30.082727,0.455799
9,REP0299,68,30.887647,0.454230


Success! Query 6 returned 500 rows.

--- Running Query 8 ---


,territory_id,month,revenue,prev_month_revenue,mom_growth_pct
0,1,1,595821.292239,NaN,NaN
1,1,2,466997.248984,595821.292239,-21.62
2,1,3,697839.366466,466997.248984,49.43
3,1,4,539300.959921,697839.366466,-22.72
4,1,5,473757.593943,539300.959921,-12.15
5,1,6,371534.401273,473757.593943,-21.58
6,1,7,386755.121759,371534.401273,4.10
7,1,8,390850.764542,386755.121759,1.06
8,1,9,542077.099709,390850.764542,38.69
9,1,10,316960.259238,542077.099709,-41.53


Success! Query 8 returned 960 rows.



In [14]:
# Q11: Rep performance percentile ranking within their specific Zone
q11 = """
WITH rep_perf AS (
    SELECT r.rep_id, t.zone, AVG(s.target_achievement_pct) as avg_ach
    FROM reps r 
    JOIN territories t USING(territory_id)
    JOIN sales s USING(territory_id)
    GROUP BY r.rep_id
)
SELECT rep_id, zone, avg_ach,
       PERCENT_RANK() OVER (PARTITION BY zone ORDER BY avg_ach) AS percentile_rank
FROM rep_perf;
"""

# Q14: Dynamic BCG Classification (Classifying territories into Star/Question Mark/etc.)
q14 = """
WITH territory_metrics AS (
    SELECT s.territory_id, 
           AVG(s.target_achievement_pct) AS avg_achievement, 
           t.market_potential_index
    FROM sales s 
    JOIN territories t USING(territory_id)
    GROUP BY s.territory_id
)
SELECT territory_id, 
       avg_achievement,
       market_potential_index,
       CASE 
           WHEN avg_achievement >= (SELECT AVG(avg_achievement) FROM territory_metrics)
                AND market_potential_index >= (SELECT AVG(market_potential_index) FROM territory_metrics) THEN 'Star'
           WHEN avg_achievement < (SELECT AVG(avg_achievement) FROM territory_metrics)
                AND market_potential_index >= (SELECT AVG(market_potential_index) FROM territory_metrics) THEN 'Question Mark'
           WHEN avg_achievement >= (SELECT AVG(avg_achievement) FROM territory_metrics)
                AND market_potential_index < (SELECT AVG(market_potential_index) FROM territory_metrics) THEN 'Cash Cow'
           ELSE 'Dog' 
       END AS bcg_quadrant
FROM territory_metrics;
"""

# Run the tests
run_test(11, q11)
run_test(14, q14)

--- Running Query 11 ---


,rep_id,zone,avg_ach,percentile_rank
0,REP0210,East,76.780203,0.000000
1,REP0399,East,76.780203,0.000000
2,REP0026,East,78.830393,0.015385
3,REP0160,East,78.830393,0.015385
4,REP0190,East,78.830393,0.015385
5,REP0243,East,78.830393,0.015385
6,REP0259,East,78.830393,0.015385
7,REP0342,East,78.830393,0.015385
8,REP0360,East,78.830393,0.015385
9,REP0451,East,78.830393,0.015385


Success! Query 11 returned 500 rows.

--- Running Query 14 ---


,territory_id,avg_achievement,market_potential_index,bcg_quadrant
0,1,78.830393,54.2,Question Mark
1,2,78.130029,62.4,Question Mark
2,3,94.385487,50.2,Cash Cow
3,4,79.911674,71.8,Question Mark
4,5,83.215310,46.0,Dog
5,6,89.036647,90.8,Star
6,7,84.303803,59.4,Question Mark
7,8,84.560146,37.1,Dog
8,9,91.802227,33.9,Cash Cow
9,10,89.284649,57.2,Star


Success! Query 14 returned 80 rows.



In [1]:
import pandas as pd
import sqlite3

# 1. Connect to your processed database
conn = sqlite3.connect('../Processed Data/processed_pharma_sfe.db')

# 2. Load the 5 core tables
sales = pd.read_sql("SELECT * FROM sales", conn)
reps = pd.read_sql("SELECT * FROM reps", conn)
territories = pd.read_sql("SELECT * FROM territories", conn)
calls = pd.read_sql("SELECT * FROM calls_cleaned", conn)
prescribers = pd.read_sql("SELECT * FROM prescribers_cleaned", conn)

conn.close()

# 3. Export as CSVs into a new 'data_for_pbix' folder (create folder manually first)
sales.to_csv('sales_fact.csv', index=False)
calls.to_csv('calls_fact.csv', index=False)
reps.to_csv('reps_dim.csv', index=False)
territories.to_csv('territories_dim.csv', index=False)
prescribers.to_csv('prescribers_dim.csv', index=False)

print("All 5 files exported successfully for Power BI!")

All 5 files exported successfully for Power BI!


In [2]:
# Create the folder manually first, then run this updated export block
sales.to_csv('../Power_BI_Data/sales_fact.csv', index=False)
calls.to_csv('../Power_BI_Data/calls_fact.csv', index=False)
reps.to_csv('../Power_BI_Data/reps_dim.csv', index=False)
territories.to_csv('../Power_BI_Data/territories_dim.csv', index=False)
prescribers.to_csv('../Power_BI_Data/prescribers_dim.csv', index=False)

print("Export complete. Files are waiting in the Power_BI_Data folder.")

Export complete. Files are waiting in the Power_BI_Data folder.


In [3]:
# 1. Run the segmentation logic again (from Phase 5)
pot_median = territories['market_potential_index'].median()
ach_median = sales.groupby('territory_id')['target_achievement_pct'].mean().median()

# Merge needed data for calculation
rep_segments = reps.merge(territories[['territory_id', 'market_potential_index']], on='territory_id')
rep_segments = rep_segments.merge(sales.groupby('territory_id')['target_achievement_pct'].mean().reset_index(), on='territory_id')

def get_segment(row):
    if row['target_achievement_pct'] >= ach_median:
        return 'Star' if row['market_potential_index'] >= pot_median else 'Cash Cow'
    else:
        return 'Question Mark' if row['market_potential_index'] >= pot_median else 'Dog'

rep_segments['performance_segment'] = rep_segments.apply(get_segment, axis=1)

# 2. Overwrite the reps_dim.csv with the new column
# Only keep the original columns + the new segment column
final_reps = rep_segments[['rep_id', 'rep_name', 'territory_id', 'tenure_months', 'monthly_salary', 'product_specialty', 'performance_segment']]
final_reps.to_csv('../Power_BI_Data/reps_dim.csv', index=False)

print("reps_dim.csv updated with performance segments!")

reps_dim.csv updated with performance segments!


In [4]:
# Check the first few rows of the CSV file we just saved
print(pd.read_csv('../Power_BI_Data/reps_dim.csv').columns)

Index(['rep_id', 'rep_name', 'territory_id', 'tenure_months', 'monthly_salary',
       'product_specialty', 'performance_segment'],
      dtype='object')
